# Notebook 1 — EDA & Preprocessing
## Diabetic Retinopathy Detection | BRSET Dataset
**Team 24 — Phase 1**  
Study Reference: Tymchenko et al. [5] — EfficientNet-B4 Backbone  
Dataset: Brazilian Multilabel Ophthalmological Dataset (BRSET)

---
### Notebook Structure
```
0. Setup & Configuration
1. Dataset Loading & Structural Audit
2. Exploratory Data Analysis (EDA)
   2.1  Label Distribution & Class Imbalance
   2.2  Multilabel Co-occurrence Analysis
   2.3  Patient-Level Analysis
   2.4  Image Quality Assessment
   2.5  Pixel Intensity & Color Space Analysis
   2.6  Representative Image Grids
3. Preprocessing Pipeline
   3.1  Circle Crop (Black Border Removal)
   3.2  Ben Graham Preprocessing
   3.3  CLAHE Enhancement
   3.4  Green Channel Normalization
   3.5  Standardized Resize (512×512)
   3.6  Pipeline Composition & Visual Comparison
4. Augmentation Strategy Preview
5. Export Preprocessed Dataset & Artifacts
```

---
## 0. Setup & Configuration

In [1]:
# ── Core dependencies ──────────────────────────────────────────────────────────
import os
import json
import warnings
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
import cv2
from tqdm.notebook import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from itertools import combinations

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [4]:
# ── Path Configuration ─────────────────────────────────────────────────────────
# Update BASE_DIR to the root of the extracted Kaggle dataset
BASE_DIR = Path('./data')                    # <-- set to your dataset root
IMAGE_DIR = BASE_DIR / 'images'             # folder containing all fundus images
LABEL_CSV = BASE_DIR / 'train.csv'         # metadata / label CSV
OUTPUT_DIR = Path('./outputs')
PREPROCESSED_DIR = OUTPUT_DIR / 'preprocessed_images'

for d in [OUTPUT_DIR, PREPROCESSED_DIR, OUTPUT_DIR / 'eda_plots', OUTPUT_DIR / 'pipeline_comparisons']:
    d.mkdir(parents=True, exist_ok=True)

# ── Global constants ───────────────────────────────────────────────────────────
TARGET_SIZE  = (512, 512)   # final image resolution fed to EfficientNet-B4
CLAHE_CLIP   = 2.0          # CLAHE clip limit
CLAHE_GRID   = (8, 8)       # CLAHE tile grid size
BEN_SIGMA    = 10           # Ben Graham Gaussian sigma
BEN_WEIGHT   = 4            # Ben Graham blend weight

print(f'Base dir     : {BASE_DIR.resolve()}')
print(f'Image dir    : {IMAGE_DIR.resolve()}')
print(f'Label CSV    : {LABEL_CSV.resolve()}')
print(f'Output dir   : {OUTPUT_DIR.resolve()}')
print(f'Target size  : {TARGET_SIZE}')

Base dir     : /home/ali/projects/XAI/Research_Project/data
Image dir    : /home/ali/projects/XAI/Research_Project/data/images
Label CSV    : /home/ali/projects/XAI/Research_Project/data/train.csv
Output dir   : /home/ali/projects/XAI/Research_Project/outputs
Target size  : (512, 512)


---
## 1. Dataset Loading & Structural Audit

In [5]:
# ── Load label CSV ─────────────────────────────────────────────────────────────
df = pd.read_csv(LABEL_CSV)
print(f'Shape        : {df.shape}')
print(f'Columns      : {list(df.columns)}')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/train.csv'

In [ ]:
# ── Structural Audit ───────────────────────────────────────────────────────────
print('=== DTYPE SUMMARY ===')
print(df.dtypes)
print('\n=== MISSING VALUES ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values detected.')
print('\n=== DUPLICATED ROWS ===')
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# ── Identify label columns automatically ──────────────────────────────────────
# BRSET label columns are binary (0/1); detect them automatically
# Adjust PATIENT_COL and IMAGE_COL to match exact column names in your CSV
PATIENT_COL = 'patient_id'     # <-- adjust if different
IMAGE_COL   = 'image_id'       # <-- adjust if different (filename stem or full path)
SPLIT_COL   = 'split'          # <-- adjust if present (train/val/test)

LABEL_COLS = [
    c for c in df.columns
    if c not in [PATIENT_COL, IMAGE_COL, SPLIT_COL]
    and df[c].dropna().isin([0, 1]).all()
]

print(f'Detected label columns ({len(LABEL_COLS)}): {LABEL_COLS}')
print(f'Total samples  : {len(df)}')
if SPLIT_COL in df.columns:
    print(f'Split counts   :')
    print(df[SPLIT_COL].value_counts())

In [ ]:
# ── Verify image files on disk ─────────────────────────────────────────────────
def resolve_image_path(image_id, image_dir):
    """Try .jpg then .png; return None if not found."""
    for ext in ['.jpg', '.jpeg', '.png']:
        p = image_dir / f'{image_id}{ext}'
        if p.exists():
            return p
    return None

df['image_path'] = df[IMAGE_COL].apply(lambda x: resolve_image_path(x, IMAGE_DIR))
missing_imgs = df['image_path'].isnull().sum()
print(f'Images found   : {len(df) - missing_imgs} / {len(df)}')
print(f'Missing images : {missing_imgs}')

# Drop rows with missing images from analysis
df = df[df['image_path'].notnull()].reset_index(drop=True)
print(f'Working dataset: {len(df)} samples')

---
## 2. Exploratory Data Analysis

### 2.1 Label Distribution & Class Imbalance

In [ ]:
# ── Per-label positive rate ────────────────────────────────────────────────────
label_counts = df[LABEL_COLS].sum().sort_values(ascending=False)
label_rates  = (label_counts / len(df) * 100).round(2)

label_summary = pd.DataFrame({
    'Positive Count': label_counts,
    'Negative Count': len(df) - label_counts,
    'Positive Rate (%)': label_rates,
    'Imbalance Ratio': (len(df) - label_counts) / label_counts.replace(0, np.nan)
})
print(label_summary.to_string())
label_summary.to_csv(OUTPUT_DIR / 'label_summary.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Absolute counts ---
colors = sns.color_palette('RdYlGn_r', len(LABEL_COLS))
bars = axes[0].barh(label_counts.index, label_counts.values, color=colors)
axes[0].set_xlabel('Positive Sample Count')
axes[0].set_title('Label Frequency (Absolute)')
for bar, val in zip(bars, label_counts.values):
    axes[0].text(val + 5, bar.get_y() + bar.get_height()/2,
                 f'{val}', va='center', fontsize=9)

# --- Positive rate % ---
axes[1].barh(label_rates.index, label_rates.values, color=colors)
axes[1].axvline(50, color='red', linestyle='--', linewidth=1, label='50% threshold')
axes[1].set_xlabel('Positive Rate (%)')
axes[1].set_title('Label Positive Rate (%)')
axes[1].legend()

plt.suptitle('BRSET — Label Distribution Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / '01_label_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Labels per sample distribution ────────────────────────────────────────────
df['label_count'] = df[LABEL_COLS].sum(axis=1)
label_count_dist  = df['label_count'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(label_count_dist.index.astype(str), label_count_dist.values,
       color=sns.color_palette('Blues_d', len(label_count_dist)))
ax.set_xlabel('Number of Active Labels per Sample')
ax.set_ylabel('Sample Count')
ax.set_title('BRSET — Multilabel Cardinality Distribution')
for x, y in zip(label_count_dist.index, label_count_dist.values):
    ax.text(str(x), y + 2, str(y), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / '02_label_cardinality.png', bbox_inches='tight')
plt.show()

print(f'\nLabel density (avg labels/sample): {df["label_count"].mean():.3f}')
print(f'Fully negative samples (no label): {(df["label_count"] == 0).sum()}')

### 2.2 Multilabel Co-occurrence Analysis

In [ ]:
# ── Co-occurrence matrix ───────────────────────────────────────────────────────
label_matrix = df[LABEL_COLS].values.astype(int)
cooccurrence = label_matrix.T @ label_matrix          # (L x L)
cooccurrence_df = pd.DataFrame(cooccurrence, index=LABEL_COLS, columns=LABEL_COLS)

# Normalize: conditional probability P(col | row)
diag = np.diag(cooccurrence).astype(float)
diag[diag == 0] = 1  # avoid division by zero
cooccurrence_norm = cooccurrence_df.values / diag[:, None]
cooccurrence_norm_df = pd.DataFrame(cooccurrence_norm, index=LABEL_COLS, columns=LABEL_COLS)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw co-occurrence
mask = np.eye(len(LABEL_COLS), dtype=bool)  # mask diagonal for raw
sns.heatmap(cooccurrence_df, ax=axes[0], annot=True, fmt='d',
            cmap='YlOrRd', linewidths=0.5, mask=mask,
            cbar_kws={'label': 'Co-occurrence Count'})
axes[0].set_title('Raw Co-occurrence Count\n(diagonal masked)')
axes[0].tick_params(axis='x', rotation=45)

# Conditional probability
np.fill_diagonal(cooccurrence_norm_df.values, np.nan)  # mask diagonal
sns.heatmap(cooccurrence_norm_df, ax=axes[1], annot=True, fmt='.2f',
            cmap='Blues', linewidths=0.5,
            cbar_kws={'label': 'P(col | row)'})
axes[1].set_title('Conditional Co-occurrence\nP(column label | row label present)')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('BRSET — Multilabel Co-occurrence Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / '03_cooccurrence_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Top co-occurring label pairs ───────────────────────────────────────────────
pair_records = []
for l1, l2 in combinations(LABEL_COLS, 2):
    both = ((df[l1] == 1) & (df[l2] == 1)).sum()
    l1_only = (df[l1] == 1).sum()
    l2_only = (df[l2] == 1).sum()
    pair_records.append({
        'Label A': l1, 'Label B': l2,
        'Co-occurrence': both,
        'P(B|A)': round(both / l1_only, 3) if l1_only else 0,
        'P(A|B)': round(both / l2_only, 3) if l2_only else 0,
    })
pair_df = pd.DataFrame(pair_records).sort_values('Co-occurrence', ascending=False)
print('Top 10 co-occurring label pairs:')
print(pair_df.head(10).to_string(index=False))
pair_df.to_csv(OUTPUT_DIR / 'label_cooccurrence_pairs.csv', index=False)

### 2.3 Patient-Level Analysis

In [ ]:
# ── Images per patient ─────────────────────────────────────────────────────────
if PATIENT_COL in df.columns:
    patient_counts = df[PATIENT_COL].value_counts()
    print(f'Total unique patients : {patient_counts.nunique()}')
    print(f'Images per patient    — mean: {patient_counts.mean():.2f}, '
          f'min: {patient_counts.min()}, max: {patient_counts.max()}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Distribution of images per patient
    axes[0].hist(patient_counts.values, bins=30, color='steelblue', edgecolor='white')
    axes[0].set_xlabel('Images per Patient')
    axes[0].set_ylabel('Number of Patients')
    axes[0].set_title('Images per Patient Distribution')

    # Patient-level label diversity
    patient_label_variety = df.groupby(PATIENT_COL)[LABEL_COLS].max().sum(axis=1)
    variety_counts = patient_label_variety.value_counts().sort_index()
    axes[1].bar(variety_counts.index.astype(str), variety_counts.values,
                color=sns.color_palette('Set2', len(variety_counts)))
    axes[1].set_xlabel('Unique Conditions per Patient')
    axes[1].set_ylabel('Patient Count')
    axes[1].set_title('Patient-level Condition Diversity')

    plt.suptitle('BRSET — Patient-Level Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'eda_plots' / '04_patient_analysis.png', bbox_inches='tight')
    plt.show()
else:
    print(f'Column "{PATIENT_COL}" not found — skipping patient-level analysis.')
    print('Update PATIENT_COL above to match your CSV.')

In [ ]:
# ── Split-level label analysis ─────────────────────────────────────────────────
if SPLIT_COL in df.columns:
    split_label_rates = df.groupby(SPLIT_COL)[LABEL_COLS].mean() * 100
    fig, ax = plt.subplots(figsize=(14, 5))
    split_label_rates.T.plot(kind='bar', ax=ax, colormap='Set1', edgecolor='white')
    ax.set_xlabel('Label')
    ax.set_ylabel('Positive Rate (%)')
    ax.set_title('BRSET — Label Positive Rate per Split (Train / Val / Test)')
    ax.legend(title='Split')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'eda_plots' / '05_split_label_rates.png', bbox_inches='tight')
    plt.show()
    print('\nSplit size summary:')
    print(df[SPLIT_COL].value_counts())

### 2.4 Image Quality Assessment

In [ ]:
# ── Quality metric functions ───────────────────────────────────────────────────
def laplacian_variance(img_bgr):
    """Blur metric: higher = sharper."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def mean_brightness(img_bgr):
    """Mean pixel brightness in HSV value channel."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    return hsv[:, :, 2].mean()

def image_contrast(img_bgr):
    """Standard deviation of grayscale — proxy for contrast."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return gray.std()

def usable_pixel_ratio(img_bgr, threshold=10):
    """Ratio of non-black pixels (fundus images have large black borders)."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return (gray > threshold).mean()

In [ ]:
# ── Compute quality metrics on a random sample ─────────────────────────────────
# For large datasets, subsample to avoid long runtime; set QUALITY_SAMPLE=len(df) for all
QUALITY_SAMPLE = min(500, len(df))
sample_df = df.sample(QUALITY_SAMPLE, random_state=SEED).copy()

quality_records = []
for _, row in tqdm(sample_df.iterrows(), total=QUALITY_SAMPLE, desc='Computing quality metrics'):
    img = cv2.imread(str(row['image_path']))
    if img is None:
        continue
    quality_records.append({
        IMAGE_COL: row[IMAGE_COL],
        'sharpness': laplacian_variance(img),
        'brightness': mean_brightness(img),
        'contrast': image_contrast(img),
        'usable_ratio': usable_pixel_ratio(img),
    })

quality_df = pd.DataFrame(quality_records)
print(quality_df.describe().round(3))

In [ ]:
# ── Quality metric visualizations ─────────────────────────────────────────────
metrics = ['sharpness', 'brightness', 'contrast', 'usable_ratio']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, metric in zip(axes, metrics):
    data = quality_df[metric].dropna()
    ax.hist(data, bins=40, color='cornflowerblue', edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(), color='crimson', linestyle='--', linewidth=1.5, label=f'Mean={data.mean():.1f}')
    ax.axvline(data.median(), color='darkorange', linestyle=':', linewidth=1.5, label=f'Median={data.median():.1f}')
    ax.set_title(f'{metric.replace("_", " ").title()} Distribution')
    ax.set_xlabel(metric)
    ax.legend(fontsize=8)

plt.suptitle(f'BRSET — Image Quality Metrics (n={QUALITY_SAMPLE})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / '06_image_quality_metrics.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Flag low-quality images ────────────────────────────────────────────────────
SHARPNESS_THRESHOLD  = quality_df['sharpness'].quantile(0.05)
BRIGHTNESS_LOW       = quality_df['brightness'].quantile(0.05)
BRIGHTNESS_HIGH      = quality_df['brightness'].quantile(0.95)
USABLE_THRESHOLD     = 0.20   # less than 20% usable pixels = heavy black border

quality_df['low_sharpness']  = quality_df['sharpness']    < SHARPNESS_THRESHOLD
quality_df['low_brightness'] = quality_df['brightness']   < BRIGHTNESS_LOW
quality_df['over_exposed']   = quality_df['brightness']   > BRIGHTNESS_HIGH
quality_df['low_usable']     = quality_df['usable_ratio'] < USABLE_THRESHOLD
quality_df['any_flag']       = quality_df[['low_sharpness','low_brightness',
                                            'over_exposed','low_usable']].any(axis=1)

print('Quality Flag Summary:')
print(quality_df[['low_sharpness','low_brightness','over_exposed','low_usable','any_flag']].sum())
print(f'\nFlagged: {quality_df["any_flag"].sum()} / {len(quality_df)} '
      f'({quality_df["any_flag"].mean()*100:.1f}%)')
quality_df.to_csv(OUTPUT_DIR / 'image_quality_report.csv', index=False)

### 2.5 Pixel Intensity & Color Space Analysis

In [ ]:
# ── Channel intensity histograms ───────────────────────────────────────────────
INTENSITY_SAMPLE = min(200, len(df))
channels = {'Red': 2, 'Green': 1, 'Blue': 0}   # OpenCV loads BGR
channel_histograms = {ch: np.zeros(256) for ch in channels}

intensity_sample = df.sample(INTENSITY_SAMPLE, random_state=SEED)
for _, row in tqdm(intensity_sample.iterrows(), total=INTENSITY_SAMPLE, desc='Channel histograms'):
    img = cv2.imread(str(row['image_path']))
    if img is None:
        continue
    for ch_name, ch_idx in channels.items():
        hist = cv2.calcHist([img], [ch_idx], None, [256], [0, 256]).flatten()
        channel_histograms[ch_name] += hist

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
ch_colors = {'Red': 'crimson', 'Green': 'seagreen', 'Blue': 'steelblue'}
for ax, (ch_name, hist) in zip(axes, channel_histograms.items()):
    ax.fill_between(range(256), hist / INTENSITY_SAMPLE, alpha=0.6, color=ch_colors[ch_name])
    ax.plot(range(256), hist / INTENSITY_SAMPLE, color=ch_colors[ch_name], linewidth=1)
    ax.set_title(f'{ch_name} Channel')
    ax.set_xlabel('Pixel Intensity')
    ax.set_ylabel('Avg Count per Image')

plt.suptitle(f'BRSET — Average Channel Intensity Distribution (n={INTENSITY_SAMPLE})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / '07_channel_intensity.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Green channel dominance analysis ──────────────────────────────────────────
# Fundus literature consistently shows green channel carries the most diagnostic info
green_dominance = []
for _, row in tqdm(intensity_sample.iterrows(), total=INTENSITY_SAMPLE, desc='Green dominance'):
    img = cv2.imread(str(row['image_path']))
    if img is None:
        continue
    b, g, r = cv2.split(img)
    total = b.mean() + g.mean() + r.mean()
    if total > 0:
        green_dominance.append({
            'R_ratio': r.mean() / total,
            'G_ratio': g.mean() / total,
            'B_ratio': b.mean() / total,
            'G_std': g.std()          # green channel variance = texture richness
        })

gd_df = pd.DataFrame(green_dominance)
print('Channel energy ratio (mean across sample):')
print(gd_df[['R_ratio','G_ratio','B_ratio']].mean().round(3))

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([gd_df['R_ratio'], gd_df['G_ratio'], gd_df['B_ratio']],
           labels=['Red', 'Green', 'Blue'],
           patch_artist=True,
           boxprops=dict(facecolor='lightyellow'),
           medianprops=dict(color='black', linewidth=2))
ax.set_ylabel('Energy Ratio')
ax.set_title('BRSET — RGB Channel Energy Ratio (fundus images)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / '08_channel_energy_ratio.png', bbox_inches='tight')
plt.show()

### 2.6 Representative Image Grids

In [ ]:
# ── Sample grid: one image per label (positive class) ─────────────────────────
def load_rgb(path, size=(256, 256)):
    img = cv2.imread(str(path))
    img = cv2.resize(img, size)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

ncols = 4
nrows = -(-len(LABEL_COLS) // ncols)   # ceiling division
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4))
axes = axes.flatten()

for idx, label in enumerate(LABEL_COLS):
    positives = df[df[label] == 1]['image_path'].dropna()
    if len(positives) == 0:
        axes[idx].axis('off')
        continue
    img_path = positives.sample(1, random_state=SEED).values[0]
    img = load_rgb(img_path)
    axes[idx].imshow(img)
    axes[idx].set_title(label.replace('_', '\n'), fontsize=9)
    axes[idx].axis('off')

for ax in axes[len(LABEL_COLS):]:
    ax.axis('off')

plt.suptitle('BRSET — Representative Positive Sample per Label', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / '09_representative_grid.png', bbox_inches='tight')
plt.show()

---
## 3. Preprocessing Pipeline

The pipeline implements four sequential transformations drawn from the DR fundus literature:

| Step | Method | Purpose |
|------|--------|---------|
| 1 | **Circle Crop** | Remove black borders; isolate retinal disc |
| 2 | **Ben Graham** | Local color normalization; reduce scanner/lighting variation |
| 3 | **CLAHE** | Enhance microstructure contrast (vessels, exudates) |
| 4 | **Green Channel Norm** | Exploit green channel's diagnostic richness |
| 5 | **Resize 512×512** | Standardize input for EfficientNet-B4 |

### 3.1 Circle Crop — Black Border Removal

In [ ]:
def circle_crop(img_bgr, scale_factor=0.97):
    """
    Detect the fundus circle and crop tightly to it.
    Uses the green channel + Otsu threshold + largest contour.
    scale_factor < 1.0 slightly pads inward to avoid artefact edges.
    """
    h, w = img_bgr.shape[:2]
    green = img_bgr[:, :, 1]                         # green channel
    blurred = cv2.GaussianBlur(green, (15, 15), 0)
    _, mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Morphological closing to fill gaps
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return img_bgr  # fallback: return original

    largest = max(contours, key=cv2.contourArea)
    x, y, cw, ch = cv2.boundingRect(largest)

    # Apply scale factor to shrink slightly inward
    cx, cy = x + cw // 2, y + ch // 2
    half_w = int(cw * scale_factor / 2)
    half_h = int(ch * scale_factor / 2)
    x1 = max(cx - half_w, 0)
    y1 = max(cy - half_h, 0)
    x2 = min(cx + half_w, w)
    y2 = min(cy + half_h, h)

    return img_bgr[y1:y2, x1:x2]

print('circle_crop() defined.')

### 3.2 Ben Graham Preprocessing

In [ ]:
def ben_graham(img_bgr, sigma=BEN_SIGMA, weight=BEN_WEIGHT):
    """
    Ben Graham's preprocessing: subtract local average color to
    normalize illumination and highlight local retinal structure.

    Output = clip(weight * img - GaussianBlur(img, sigma) + 128)
    """
    blurred = cv2.GaussianBlur(img_bgr, (0, 0), sigma)
    processed = cv2.addWeighted(img_bgr, weight, blurred, -(weight - 1), 128)
    return np.clip(processed, 0, 255).astype(np.uint8)

print('ben_graham() defined.')

### 3.3 CLAHE Enhancement

In [ ]:
def apply_clahe(img_bgr, clip_limit=CLAHE_CLIP, tile_grid=CLAHE_GRID):
    """
    Apply CLAHE independently to each BGR channel.
    Operates per-channel (not on grayscale conversion) to preserve
    color-channel diagnostic information.
    """
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    channels = cv2.split(img_bgr)
    enhanced = [clahe.apply(ch) for ch in channels]
    return cv2.merge(enhanced)

print('apply_clahe() defined.')

### 3.4 Green Channel Normalization

In [ ]:
def green_channel_normalize(img_bgr):
    """
    Scale all channels by the inverse mean of the green channel.
    This equalizes inter-image brightness variation using the most
    diagnostically informative channel as the reference anchor.
    """
    green_mean = img_bgr[:, :, 1].mean()
    if green_mean == 0:
        return img_bgr
    scale = 128.0 / green_mean
    normalized = np.clip(img_bgr.astype(np.float32) * scale, 0, 255)
    return normalized.astype(np.uint8)

print('green_channel_normalize() defined.')

### 3.5 Resize

In [ ]:
def resize_image(img_bgr, target=TARGET_SIZE, interpolation=cv2.INTER_LANCZOS4):
    """
    Resize to target size using Lanczos4 (highest quality downsampling).
    Preferred over bilinear for medical images where fine structures matter.
    """
    return cv2.resize(img_bgr, target, interpolation=interpolation)

print('resize_image() defined.')

### 3.6 Pipeline Composition & Visual Comparison

In [ ]:
def full_pipeline(img_bgr):
    """
    Full sequential preprocessing pipeline.
    Returns the preprocessed uint8 BGR image (512×512).
    """
    img = circle_crop(img_bgr)         # Step 1: remove black borders
    img = ben_graham(img)              # Step 2: illumination normalization
    img = apply_clahe(img)             # Step 3: contrast enhancement
    img = green_channel_normalize(img) # Step 4: green channel anchoring
    img = resize_image(img)            # Step 5: standardize resolution
    return img

print('full_pipeline() defined.')

In [ ]:
# ── Step-by-step visual comparison on representative samples ──────────────────
COMPARE_N = 3   # number of images to visualize in the comparison grid
step_names = ['Original', 'Circle Crop', 'Ben Graham', 'CLAHE', 'Green Norm', 'Final (512²)']

compare_paths = df.dropna(subset=['image_path']).sample(COMPARE_N, random_state=SEED)['image_path'].values

fig, axes = plt.subplots(COMPARE_N, len(step_names), figsize=(len(step_names) * 3.5, COMPARE_N * 3.5))

for row_idx, img_path in enumerate(compare_paths):
    raw = cv2.imread(str(img_path))
    if raw is None:
        continue

    step_imgs = [
        raw,
        circle_crop(raw),
        ben_graham(circle_crop(raw)),
        apply_clahe(ben_graham(circle_crop(raw))),
        green_channel_normalize(apply_clahe(ben_graham(circle_crop(raw)))),
        full_pipeline(raw)
    ]

    for col_idx, (step_img, step_name) in enumerate(zip(step_imgs, step_names)):
        ax = axes[row_idx, col_idx] if COMPARE_N > 1 else axes[col_idx]
        display = cv2.cvtColor(cv2.resize(step_img, (256, 256)), cv2.COLOR_BGR2RGB)
        ax.imshow(display)
        if row_idx == 0:
            ax.set_title(step_name, fontsize=9, fontweight='bold')
        ax.axis('off')

plt.suptitle('BRSET — Step-by-Step Preprocessing Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pipeline_comparisons' / '01_preprocessing_steps.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Pixel statistics: before vs. after preprocessing ──────────────────────────
STAT_SAMPLE = min(100, len(df))
stat_sample = df.dropna(subset=['image_path']).sample(STAT_SAMPLE, random_state=SEED)

before_means, after_means = [], []
before_stds,  after_stds  = [], []

for _, row in tqdm(stat_sample.iterrows(), total=STAT_SAMPLE, desc='Before/after stats'):
    raw = cv2.imread(str(row['image_path']))
    if raw is None:
        continue
    proc = full_pipeline(raw)
    before_means.append(raw.mean())
    after_means.append(proc.mean())
    before_stds.append(raw.std())
    after_stds.append(proc.std())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(before_means, bins=30, alpha=0.6, color='salmon', label='Before', edgecolor='white')
axes[0].hist(after_means,  bins=30, alpha=0.6, color='mediumseagreen', label='After', edgecolor='white')
axes[0].set_title('Mean Pixel Intensity: Before vs After')
axes[0].set_xlabel('Mean Pixel Value')
axes[0].legend()

axes[1].hist(before_stds, bins=30, alpha=0.6, color='salmon', label='Before', edgecolor='white')
axes[1].hist(after_stds,  bins=30, alpha=0.6, color='mediumseagreen', label='After', edgecolor='white')
axes[1].set_title('Pixel Std Dev: Before vs After')
axes[1].set_xlabel('Std Dev')
axes[1].legend()

plt.suptitle('Effect of Preprocessing Pipeline on Pixel Statistics', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pipeline_comparisons' / '02_before_after_stats.png', bbox_inches='tight')
plt.show()

---
## 4. Augmentation Strategy Preview
Augmentations are applied **only at training time** inside the PyTorch DataLoader in Notebook 2.  
This section previews their visual effect on preprocessed images.

In [ ]:
# ── Augmentation functions (preview only — not applied to saved files) ─────────
def aug_hflip(img):     return cv2.flip(img, 1)
def aug_vflip(img):     return cv2.flip(img, 0)
def aug_rotate(img, angle=25):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LANCZOS4,
                          borderMode=cv2.BORDER_REFLECT)
def aug_brightness(img, factor=1.3):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:,:,2] = np.clip(hsv[:,:,2] * factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
def aug_cutout(img, n=3, size=50):
    out = img.copy()
    h, w = img.shape[:2]
    for _ in range(n):
        x = random.randint(0, w - size)
        y = random.randint(0, h - size)
        out[y:y+size, x:x+size] = 0
    return out
def aug_elastic(img, alpha=500, sigma=20):
    """Elastic deformation — simulates retinal image variation."""
    h, w = img.shape[:2]
    dx = cv2.GaussianBlur(np.random.randn(h, w).astype(np.float32), (0,0), sigma) * alpha
    dy = cv2.GaussianBlur(np.random.randn(h, w).astype(np.float32), (0,0), sigma) * alpha
    x, y = np.meshgrid(np.arange(w), np.arange(h))
    map_x = np.clip(x + dx, 0, w-1).astype(np.float32)
    map_y = np.clip(y + dy, 0, h-1).astype(np.float32)
    return cv2.remap(img, map_x, map_y, cv2.INTER_LANCZOS4)

print('Augmentation functions defined.')

In [ ]:
# ── Augmentation grid visualization ───────────────────────────────────────────
aug_funcs = {
    'Original (preprocessed)': lambda x: x,
    'Horizontal Flip':          aug_hflip,
    'Vertical Flip':            aug_vflip,
    'Rotation +25°':            lambda x: aug_rotate(x, 25),
    'Rotation −25°':            lambda x: aug_rotate(x, -25),
    'Brightness ×1.3':          aug_brightness,
    'Cutout (n=3)':             aug_cutout,
    'Elastic Deform':           aug_elastic,
}

AUG_IMGS = min(2, len(df))
aug_sample_paths = df.dropna(subset=['image_path']).sample(AUG_IMGS, random_state=SEED)['image_path'].values

fig, axes = plt.subplots(AUG_IMGS, len(aug_funcs), figsize=(len(aug_funcs)*3, AUG_IMGS*3))
if AUG_IMGS == 1:
    axes = axes[np.newaxis, :]

for row_i, path in enumerate(aug_sample_paths):
    raw = cv2.imread(str(path))
    preprocessed = full_pipeline(raw)
    for col_i, (aug_name, aug_fn) in enumerate(aug_funcs.items()):
        augmented = aug_fn(preprocessed.copy())
        display   = cv2.cvtColor(augmented, cv2.COLOR_BGR2RGB)
        axes[row_i, col_i].imshow(display)
        if row_i == 0:
            axes[row_i, col_i].set_title(aug_name, fontsize=8, fontweight='bold')
        axes[row_i, col_i].axis('off')

plt.suptitle('BRSET — Augmentation Strategy Preview (applied to preprocessed images)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pipeline_comparisons' / '03_augmentation_preview.png', bbox_inches='tight')
plt.show()

---
## 5. Export Preprocessed Dataset & Artifacts

In [ ]:
# ── Process and save all images ────────────────────────────────────────────────
# This saves preprocessed images to PREPROCESSED_DIR as .jpg
# Notebook 2 will load from this directory instead of the raw images

SAVE_QUALITY = 95   # JPEG compression quality
errors = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Preprocessing & saving'):
    raw = cv2.imread(str(row['image_path']))
    if raw is None:
        errors.append(row[IMAGE_COL])
        continue
    try:
        proc = full_pipeline(raw)
        out_path = PREPROCESSED_DIR / f'{row[IMAGE_COL]}.jpg'
        cv2.imwrite(str(out_path), proc, [cv2.IMWRITE_JPEG_QUALITY, SAVE_QUALITY])
    except Exception as e:
        errors.append(row[IMAGE_COL])

print(f'Saved : {len(df) - len(errors)} / {len(df)}')
print(f'Errors: {len(errors)}')
if errors:
    print('Failed IDs:', errors[:10])

In [ ]:
# ── Export cleaned label CSV ───────────────────────────────────────────────────
# Add preprocessed path column and drop raw image_path
df['preprocessed_path'] = df[IMAGE_COL].apply(
    lambda x: str(PREPROCESSED_DIR / f'{x}.jpg')
)
df_export = df.drop(columns=['image_path'])
df_export.to_csv(OUTPUT_DIR / 'labels_preprocessed.csv', index=False)
print(f'Exported labels_preprocessed.csv — {len(df_export)} rows, {len(df_export.columns)} columns')
df_export.head()

In [ ]:
# ── Compute and save per-label Focal Loss weights ──────────────────────────────
# These weights are consumed by Notebook 2 to initialize the loss function
# Formula: pos_weight_i = (N - n_i) / n_i  (standard BCEWithLogitsLoss approach)
pos_counts = df[LABEL_COLS].sum()
N          = len(df)
pos_weights = ((N - pos_counts) / pos_counts.replace(0, 1)).round(4)

pos_weight_dict = pos_weights.to_dict()
with open(OUTPUT_DIR / 'pos_weights.json', 'w') as f:
    json.dump(pos_weight_dict, f, indent=2)

print('Positive weights for BCEWithLogitsLoss / Focal Loss:')
for label, w in pos_weight_dict.items():
    print(f'  {label:<35} {w:.4f}')

In [ ]:
# ── Save pipeline config for reproducibility ──────────────────────────────────
pipeline_config = {
    'target_size': list(TARGET_SIZE),
    'clahe_clip_limit': CLAHE_CLIP,
    'clahe_tile_grid': list(CLAHE_GRID),
    'ben_graham_sigma': BEN_SIGMA,
    'ben_graham_weight': BEN_WEIGHT,
    'circle_crop_scale': 0.97,
    'jpeg_quality': SAVE_QUALITY,
    'label_columns': LABEL_COLS,
    'n_samples_total': len(df),
    'seed': SEED
}
with open(OUTPUT_DIR / 'pipeline_config.json', 'w') as f:
    json.dump(pipeline_config, f, indent=2)
print('Pipeline config saved to outputs/pipeline_config.json')

In [ ]:
# ── Summary report ─────────────────────────────────────────────────────────────
print('=' * 60)
print('  NOTEBOOK 1 — COMPLETION SUMMARY')
print('=' * 60)
print(f'  Dataset samples        : {len(df)}')
print(f'  Label columns          : {len(LABEL_COLS)}')
print(f'  Preprocessed images    : {len(df) - len(errors)}')
print(f'  Output directory       : {OUTPUT_DIR.resolve()}')
print()
print('  Artifacts exported:')
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():
        print(f'    {f.relative_to(OUTPUT_DIR)}')
print()
print('  Next: Open notebook_02_model_training.ipynb')
print('  Inputs for Notebook 2:')
print('    - outputs/labels_preprocessed.csv')
print('    - outputs/preprocessed_images/')
print('    - outputs/pos_weights.json')
print('    - outputs/pipeline_config.json')